# Managing Agent State - Saving and Loading Agents

In [16]:
import asyncio
from autogen_core import CancellationToken
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
model_client = OpenAIChatCompletionClient(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
    model_info={
        "family": "gpt-4o",
        "vision": True,
        "function_calling": True,
        "json_output": True,
        "structured_output": True,
    },
)

In [17]:
assistant = AssistantAgent(
    name = 'Assistant',
    model_client = model_client,
    system_message = 'You are a helpful assistant who can answer questions and provide information on a wide range of topics. You are friendly, informative, and concise.'
)

response = await assistant.on_messages(
    [TextMessage(content = 'Can you help me with a travel itinerary of France from May to June? Keep it under 30 words.',
    source = 'user')],
    cancellation_token = CancellationToken(),
)

print(response.chat_message.content)

Paris, Versailles, Loire Valley, Bordeaux, Lyon, Nice, Cannes, Mont Saint‑Michel, Marseille, Strasbourg.


In [7]:
agent_state = await assistant.save_state()

In [11]:
print(agent_state)

{'type': 'AssistantAgentState', 'version': '1.0.0', 'llm_context': {'messages': [{'content': 'Can you help me with a travel itinerary of France from May to June? Keep it under 30 words.', 'source': 'user', 'type': 'UserMessage'}, {'content': 'May: Paris (3\u202fdays), Versailles (1), Giverny (1), Lyon (2).  \nJune: Bordeaux (3), Dordogne (2), Nice (4), Marseille (2).', 'thought': None, 'source': 'Assistant', 'type': 'AssistantMessage'}]}}


In [12]:
new_assistant_agent = AssistantAgent(
    name = 'Assistant_agent_2',
    model_client = model_client,
    system_message = 'You are a helpful assistant.'
)

In [13]:
await new_assistant_agent.load_state(agent_state)

In [14]:

response = await new_assistant_agent.on_messages(
    [TextMessage(content = 'Can you tell me what was the last discussion about?',
    source = 'user')],
    cancellation_token = CancellationToken(),
)
print(response.chat_message)

source='Assistant_agent_2' models_usage=RequestUsage(prompt_tokens=168, completion_tokens=106) metadata={} content='The last request was for a concise (under\u202f30\u202fwords) travel itinerary for France covering May to June. I provided a 2‑week plan with key destinations in Paris, Lyon, Bordeaux, Nice, and Marseille.' type='TextMessage'


In [15]:
response.chat_message.content

'The last request was for a concise (under\u202f30\u202fwords) travel itinerary for France covering May to June. I provided a 2‑week plan with key destinations in Paris, Lyon, Bordeaux, Nice, and Marseille.'